# Homework 3 analysis

### Import packages

In [1]:
import os, wrds, fredapi, math
import pandas as pd, numpy as np, statsmodels.api as sm
pd.options.mode.chained_assignment = None  # default='warn'
from scipy.optimize import minimize
from matplotlib import pyplot as plt

### Import data

Establish WRDS connection

In [2]:
conn = wrds.Connection(wrds_username='wmann')

Loading library list...
Done


One-month risk-free returns

In [3]:
FF3F = conn.get_table(library='ff_all',table='factors_monthly')
FF3F['month'] = pd.to_datetime(FF3F['date']).dt.to_period('M')
FF3F = FF3F.drop('date',axis=1).drop('dateff',axis=1).set_index('month')
riskfree = FF3F.rf.astype('float64')
riskfree = riskfree['2005':'2024']

Mutual fund returns

In [4]:
tickers = ('VFINX','FBDIX','FKUTX','PTLAX','VMNFX')
identifiers = conn.raw_sql( "select crsp_fundno, ticker from crsp_q_mutualfunds.fund_hdr " \
        + "where ticker in " + "('" + "','".join(tickers) + "')").set_index('crsp_fundno')
fund_returns = conn.raw_sql( "select crsp_fundno, caldt, mret from crsp_q_mutualfunds.monthly_returns " \
        + "where crsp_fundno in " + "('" + "','".join( str(int(identifier)) for identifier in identifiers.index.values ) + "')")

# Convert to monthly frequency
fund_returns['month'] = pd.to_datetime(fund_returns['caldt']).dt.to_period('M')
fund_returns = fund_returns.set_index(['crsp_fundno','month']).drop('caldt',axis=1)

# Merge tickers back on
fund_returns = identifiers.join(fund_returns).reset_index().drop('crsp_fundno',axis=1).set_index(['ticker','month'])

# Convert to a dictionary, which is easier to work with in many ways
fund_returns = {ticker: fund_returns.loc[ticker].mret for ticker in tickers}

# Keep just the dates I want
fund_returns = {ticker: fund['2005':'2024'] for ticker,fund in fund_returns.items()}

Save data to Excel

In [5]:
# try: os.remove("Homework 3 data.xlsx")
# except OSError: pass

# # set up custom ExcelWriter engine to get the date format correct in the output file
# writer = pd.ExcelWriter('Homework 3 data.xlsx', engine='xlsxwriter', datetime_format= "yyyy-mm")

# riskfree.to_excel(writer,sheet_name='Data',index_label='Month',header=['Risk-free return'])

# for (index,ticker) in enumerate(tickers):
#     fund_returns[ticker].to_excel(writer,sheet_name='Data',index=False,header=[ticker+" return"],startcol=index+2)

# writer.close()

Close database connection

In [6]:
conn.close()

### Analysis

Create a list of excess returns.

In [7]:
fund_excess_returns = { ticker: fund_returns[ticker] - riskfree for ticker in tickers }

Calculate and print average excess returns.

In [8]:
for ticker in tickers:
    print("Ticker: " + ticker)
    print(f"  Average return: {100*fund_excess_returns[ticker].mean():5.4f}%")

Ticker: VFINX
  Average return: 0.7800%
Ticker: FBDIX
  Average return: 0.7691%
Ticker: FKUTX
  Average return: 0.6744%
Ticker: PTLAX
  Average return: 0.0702%
Ticker: VMNFX
  Average return: 0.1231%


Calculate and print volatility of excess return.

In [9]:
for ticker in tickers:
    print("Ticker: " + ticker)
    print(f" Volatility of excess return: {100*fund_excess_returns[ticker].std():5.4f}%")

Ticker: VFINX
 Volatility of excess return: 4.3340%
Ticker: FBDIX
 Volatility of excess return: 6.0993%
Ticker: FKUTX
 Volatility of excess return: 3.9441%
Ticker: PTLAX
 Volatility of excess return: 0.7446%
Ticker: VMNFX
 Volatility of excess return: 1.6963%


1. Which funds, if any, had Sharpe ratios higher than VFINX in the data?

In [10]:
for ticker in tickers:
    print("Ticker: " + ticker)
    Sharpe_ratio = fund_excess_returns[ticker].mean() / fund_excess_returns[ticker].std()
    print(f"  Sharpe ratio: {Sharpe_ratio:5.4f}")

Ticker: VFINX
  Sharpe ratio: 0.1800
Ticker: FBDIX
  Sharpe ratio: 0.1261
Ticker: FKUTX
  Sharpe ratio: 0.1710
Ticker: PTLAX
  Sharpe ratio: 0.0943
Ticker: VMNFX
  Sharpe ratio: 0.0726


Answer: None of the funds had a higher Sharpe ratio than VFINX.

2. Calculate the beta and alpha of each fund with respect to VFINX, using the formulas from class.

In [11]:
for ticker in tickers:
    print(ticker + ": ")
    correlation = np.corrcoef([fund_excess_returns[ticker],fund_excess_returns['VFINX']])[0,1]
    vol_fund = fund_excess_returns[ticker].std()
    vol_VFINX = fund_excess_returns['VFINX'].std()
    beta = correlation*vol_fund/vol_VFINX
    print(f"  Beta: {beta:5.4f}")
    alpha = fund_excess_returns[ticker].mean() - beta*fund_excess_returns['VFINX'].mean()
    print(f"  Alpha: {100*alpha:5.4f}%" )

VFINX: 
  Beta: 1.0000
  Alpha: 0.0000%
FBDIX: 
  Beta: 0.8329
  Alpha: 0.1194%
FKUTX: 
  Beta: 0.5141
  Alpha: 0.2733%
PTLAX: 
  Beta: 0.0682
  Alpha: 0.0171%
VMNFX: 
  Beta: -0.0026
  Alpha: 0.1252%


3. Calculate the same numbers by running a regression and reporting the results.

In [12]:
for ticker in tickers:
    print(ticker + ":")
    (alpha,beta) = sm.formula.ols(ticker + " ~ VFINX",data=fund_excess_returns).fit().params
    print(f"  Beta: {beta:5.4f}")
    alpha = fund_excess_returns[ticker].mean() - beta*fund_excess_returns['VFINX'].mean()
    print(f"  Alpha: {100*alpha:5.4f}%" )

VFINX:
  Beta: 1.0000
  Alpha: 0.0000%
FBDIX:
  Beta: 0.8329
  Alpha: 0.1194%
FKUTX:
  Beta: 0.5141
  Alpha: 0.2733%
PTLAX:
  Beta: 0.0682
  Alpha: 0.0171%
VMNFX:
  Beta: -0.0026
  Alpha: 0.1252%


4. For each fund, calculate its information ratio with respect to VFINX. Then determine the highest Sharpe ratio that an investor could achieve by allocating only between VFINX and that fund. (Refer to the formulas from class.)

In [13]:
SR_VFINX = fund_excess_returns['VFINX'].mean() / fund_excess_returns['VFINX'].std()
print(f"VFINX Sharpe ratio: {SR_VFINX:5.4f}")
print("")

for ticker in tickers:
        if ticker == "VFINX": continue
        print(ticker + ":")
        OLS_results = sm.formula.ols(ticker + " ~ VFINX",data=fund_excess_returns).fit()
        alpha = OLS_results.params['Intercept']
        resid_std = OLS_results.resid.std()
        IR = alpha / resid_std
        print(f"  Information ratio: {IR:5.4f}" )
        Max_SR = np.sqrt( SR_VFINX**2 + IR**2 )
        print(f"  Max attainable Sharpe ratio: {Max_SR:5.4f}")

VFINX Sharpe ratio: 0.1800

FBDIX:
  Information ratio: 0.0243
  Max attainable Sharpe ratio: 0.1816
FKUTX:
  Information ratio: 0.0840
  Max attainable Sharpe ratio: 0.1986
PTLAX:
  Information ratio: 0.0250
  Max attainable Sharpe ratio: 0.1817
VMNFX:
  Information ratio: 0.0738
  Max attainable Sharpe ratio: 0.1945


5. Suppose a fund manager follows a strategy of allocating half of invested funds to FKUTX, and the other half to VMNFX (rebalanced every month). What is the information ratio of this strategy? If an investor divides her money between this fund and VFINX, what is the highest Sharpe ratio that she can achieve? What weighting of VFINX and the manager's strategy will achieve this Sharpe ratio?

In [14]:
fund_excess_returns['Combo'] = 0.5*fund_excess_returns['VMNFX'] + 0.5*fund_excess_returns['FKUTX']
OLS_results = sm.formula.ols("Combo ~ VFINX",data=fund_excess_returns).fit()
IR = OLS_results.params['Intercept'] / OLS_results.resid.std()
print(f"Information ratio: {IR:5.4f}")
print("")

Max_SR = np.sqrt( SR_VFINX**2 + IR**2 )
print(f"Max attainable Sharpe ratio: {Max_SR:6.5f}")
print("")

Optimal_weights_unscaled = np.linalg.inv( np.cov([fund_excess_returns['Combo'],fund_excess_returns['VFINX']]) ) @ [fund_excess_returns['Combo'].mean(),fund_excess_returns['VFINX'].mean()]
Optimal_weights = Optimal_weights_unscaled / Optimal_weights_unscaled.sum()
print("The weights that achieve this are: " )
print(f"  {100*Optimal_weights[0]:4.3f}% on the active portfolio,")
print(f"  {100*Optimal_weights[1]:4.3f}% on VFINX")
print("")

print("To check this, calculate the Sharpe ratio with these weights: ")
avg = Optimal_weights @ [fund_excess_returns['Combo'].mean() , fund_excess_returns['VFINX'].mean()]
var = Optimal_weights @ ( np.cov([fund_excess_returns['Combo'],fund_excess_returns['VFINX']]) @ Optimal_weights )
Sharpe_ratio = avg / np.sqrt(var)
print(f"  Sharpe ratio: {Sharpe_ratio:6.5f}" )
print("As we would hope, this exactly matches the Sharpe ratio that we calculated earlier.")

Information ratio: 0.1069

Max attainable Sharpe ratio: 0.20935

The weights that achieve this are: 
  68.126% on the active portfolio,
  31.874% on VFINX

To check this, calculate the Sharpe ratio with these weights: 
  Sharpe ratio: 0.20935
As we would hope, this exactly matches the Sharpe ratio that we calculated earlier.
